# HybridBid — Day 1: ERCOT Data Exploration

**Goal:** Understand the actual data landscape before committing to a pipeline design.

Checklist:
1. Install gridstatus, inspect available methods
2. Pull sample data for each key product (pre and post RTC+B)
3. Document column names, granularity, gaps
4. Compare formats across the Dec 5, 2025 boundary
5. Download ESR name mapping file

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', 200)

# Key dates
RTCB_DATE = '2025-12-05'
print(f'RTC+B Go-Live: {RTCB_DATE}')

## 1. gridstatus Discovery

Explore what methods and data products are available.

In [ ]:
import gridstatus
print(f'gridstatus version: {gridstatus.__version__}')

# Scraper client (no auth needed)
ercot = gridstatus.Ercot()

# List all get_* methods
methods = [m for m in dir(ercot) if m.startswith('get_')]
for m in sorted(methods):
    doc = getattr(ercot, m).__doc__
    first_line = doc.strip().split('\n')[0] if doc else '(no docstring)'
    print(f'  .{m}() — {first_line}')

In [ ]:
# Try the API client (needs credentials)
try:
    api = gridstatus.ErcotAPI()
    api_methods = [m for m in dir(api) if m.startswith('get_')]
    print(f'ErcotAPI methods: {len(api_methods)}')
    for m in sorted(api_methods):
        print(f'  .{m}()')
except Exception as e:
    print(f'ErcotAPI not available: {e}')
    print('Register at https://apiexplorer.ercot.com for free API access')

## 2. Energy Prices — RT SPP

Pull sample data from both pre and post RTC+B periods.

In [ ]:
# Pre-RTC+B sample: 1 week in November 2025
print('--- Pre-RTC+B RT SPP ---')
rt_spp_pre = ercot.get_spp(
    date='2025-11-01',
    end='2025-11-08',
    market='REAL_TIME_15_MIN',
    verbose=True,
)
print(f'Shape: {rt_spp_pre.shape}')
print(f'Columns: {rt_spp_pre.columns.tolist()}')
print(f'Index type: {type(rt_spp_pre.index)}')
print(f'Date range: {rt_spp_pre.index.min()} to {rt_spp_pre.index.max()}')
rt_spp_pre.head()

In [ ]:
# Post-RTC+B sample: 1 week in January 2026
print('--- Post-RTC+B RT SPP ---')
rt_spp_post = ercot.get_spp(
    date='2026-01-05',
    end='2026-01-12',
    market='REAL_TIME_15_MIN',
    verbose=True,
)
print(f'Shape: {rt_spp_post.shape}')
print(f'Columns: {rt_spp_post.columns.tolist()}')
rt_spp_post.head()

In [ ]:
# CRITICAL: Compare column names/structure across the boundary
print('Pre-RTC+B columns:', sorted(rt_spp_pre.columns.tolist()))
print('Post-RTC+B columns:', sorted(rt_spp_post.columns.tolist()))

# Check for differences
pre_cols = set(rt_spp_pre.columns)
post_cols = set(rt_spp_post.columns)
print(f'\nNew in post-RTC+B: {post_cols - pre_cols}')
print(f'Removed in post-RTC+B: {pre_cols - post_cols}')

## 3. Ancillary Service Prices

DAM AS prices (full history) and RT MCPCs (post-RTC+B only).

In [ ]:
# DAM AS prices
print('--- DAM AS Prices ---')
try:
    dam_as = ercot.get_as_prices(
        date='2025-11-01',
        end='2025-11-08',
        verbose=True,
    )
    print(f'Shape: {dam_as.shape}')
    print(f'Columns: {dam_as.columns.tolist()}')
    dam_as.head()
except Exception as e:
    print(f'Failed: {e}')
    print('Try alternative method names...')

In [ ]:
# RT AS MCPCs (post-RTC+B only)
# This is the NEW data product NP6-332-CD.
# Method name in gridstatus TBD — explore here.
print('--- RT AS MCPCs (post-RTC+B) ---')

# Try various possible method names
for method_name in ['get_as_mcpc', 'get_rt_as_clearing_prices', 
                     'get_as_clearing_prices', 'get_mcpc']:
    if hasattr(ercot, method_name):
        print(f'Found method: {method_name}')
        func = getattr(ercot, method_name)
        try:
            rt_as = func(date='2026-01-05', end='2026-01-08')
            print(f'Shape: {rt_as.shape}')
            print(f'Columns: {rt_as.columns.tolist()}')
            display(rt_as.head())
            break
        except Exception as e:
            print(f'  Error: {e}')

# If not found via scraper, try the API client
print('\nTrying via ErcotAPI...')
try:
    api = gridstatus.ErcotAPI()
    rt_as_api = api.get_data(
        report_type_id='NP6-332-CD',
        start='2026-01-05',
        end='2026-01-08',
    )
    print(f'Shape: {rt_as_api.shape}')
    print(f'Columns: {rt_as_api.columns.tolist()}')
except Exception as e:
    print(f'API fetch failed: {e}')

## 4. System Conditions — Load, Wind, Solar

In [ ]:
# Load data
print('--- Load Data ---')
load = ercot.get_load(date='2025-11-01', end='2025-11-08', verbose=True)
print(f'Shape: {load.shape}')
print(f'Columns: {load.columns.tolist()}')
print(f'Granularity: {load.index.to_series().diff().median()}')
load.head()

In [ ]:
# Fuel mix
print('--- Fuel Mix ---')
fuel = ercot.get_fuel_mix(date='2025-11-01', end='2025-11-08', verbose=True)
print(f'Shape: {fuel.shape}')
print(f'Columns: {fuel.columns.tolist()}')
fuel.head()

## 5. Data Availability Audit

Test access across the full date range we need.

In [ ]:
# Quick test: can we access 2020 data?
print('--- Historical Access Test ---')
try:
    old_data = ercot.get_spp(
        date='2020-01-01',
        end='2020-01-03',
        market='REAL_TIME_15_MIN',
        verbose=True,
    )
    print(f'2020 data accessible: {len(old_data)} rows')
except Exception as e:
    print(f'2020 data NOT accessible via scraper: {e}')
    print('Will need bulk file downloads for 2020-2023')

## 6. Column Mapping Notes

**Fill this in after running the cells above.** These mappings drive the preprocessing module.

```
RT SPP columns:     [TODO: fill in]
DAM SPP columns:    [TODO: fill in]
DAM AS columns:     [TODO: fill in]
RT MCPC columns:    [TODO: fill in]
Load columns:       [TODO: fill in]
Fuel mix columns:   [TODO: fill in]

Post-RTC+B changes: [TODO: document any column name changes]
```

→ Update `src/data/preprocessing.py` column mappings based on these findings.

## 7. Quick Visualization — Price Patterns

In [ ]:
# If we have RT SPP data, plot a sample week
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

if 'rt_spp_pre' in dir() and not rt_spp_pre.empty:
    # Find the main price column
    price_cols = [c for c in rt_spp_pre.columns if 'SPP' in str(c).upper() or 'LMP' in str(c).upper()]
    if price_cols:
        axes[0].plot(rt_spp_pre.index, rt_spp_pre[price_cols[0]], linewidth=0.5)
        axes[0].set_title(f'Pre-RTC+B Energy Prices ({price_cols[0]})')
        axes[0].set_ylabel('$/MWh')

if 'rt_spp_post' in dir() and not rt_spp_post.empty:
    price_cols = [c for c in rt_spp_post.columns if 'SPP' in str(c).upper() or 'LMP' in str(c).upper()]
    if price_cols:
        axes[1].plot(rt_spp_post.index, rt_spp_post[price_cols[0]], linewidth=0.5, color='orange')
        axes[1].set_title(f'Post-RTC+B Energy Prices ({price_cols[0]})')
        axes[1].set_ylabel('$/MWh')

plt.tight_layout()
plt.savefig('../data/raw/price_exploration.png', dpi=150)
plt.show()